<a href="https://colab.research.google.com/github/heyanugrah/sentimentAnalysisTransformerEncoder/blob/main/mytransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Basic Transformer- Encoder

In [18]:
import torch
import torch.nn as nn
import math

In [19]:
# 1. Define the Vocabulary (Adding '[sep]')
new_vocab = {'<pad>': 0, '<unk>': 1, '<cls>': 2, '[sep]': 3, 'this': 4, 'is': 5, 'a': 6, 'positive': 7, 'sentence': 8, 'learning': 9, 'about': 10, 'transformers': 11, 'fun': 12, 'i': 13, 'hate': 14, 'am': 15, 'so': 16, 'disappointed': 17, 'really': 18, 'enjoyed': 19, 'movie': 20, 'wonderful': 21, 'day': 22, 'for': 23, 'walk': 24, 'product': 25, 'excellent!': 26, 'absolutely': 27, 'dreadful': 28, 'service': 29, 'what': 30, 'terrible': 31, 'experience': 32, 'feeling': 33, 'very': 34, 'happy': 35, 'today': 36, 'boy': 37, 'anugrah': 38}

# 2. Define the Reverse Vocabulary
idx_to_word = {i: word for word, i in new_vocab.items()}

vocab_size = len(new_vocab)

print(f"Vocabulary size is: {vocab_size}")

Vocabulary size is: 39


### Modular Tokenization Functions

To improve clarity and reusability, let's break down the tokenization process into several distinct functions:

1.  `_add_special_tokens_and_segments`: Adds `<cls>` and `[sep]` tokens and initializes segment IDs.
2.  `_convert_to_ids`: Converts token strings to their numerical IDs.
3.  `_pad_ids_and_segments`: Handles padding for both token IDs and segment IDs.
4.  `_create_attention_mask_from_ids`: Generates the attention mask from padded token IDs.
5.  `prepare_transformer_inputs`: An orchestrating function that calls the above helpers to produce all necessary inputs for a Transformer model.

In [20]:
# Helper to add special tokens and initial segment IDs
def _add_special_tokens_and_segments(input_text, vocab):
    tokens = [vocab.get("<cls>", "<cls>")] # Start with <cls>
    segment_ids = [0] # <cls> belongs to segment 0

    # Tokenize input text and add words
    for word in input_text.lower().replace('.', '').split():
        tokens.append(word)
        segment_ids.append(0) # All words in the first sentence belong to segment 0

    tokens.append(vocab.get("[sep]", "[sep]")) # Add [sep]
    segment_ids.append(0) # [sep] belongs to segment 0

    return tokens, segment_ids

# Helper to convert tokens to IDs
def _convert_to_ids(tokens, vocab):
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

# Helper to pad token IDs and segment IDs
def _pad_ids_and_segments(token_ids, segment_ids, max_seq_len, pad_token_id):
    if len(token_ids) > max_seq_len:
        token_ids = token_ids[:max_seq_len]
        segment_ids = segment_ids[:max_seq_len]
    else:
        padding_length = max_seq_len - len(token_ids)
        token_ids.extend([pad_token_id] * padding_length)
        segment_ids.extend([0] * padding_length) # Padding tokens belong to segment 0

    return token_ids, segment_ids

# Helper to create attention mask
def _create_attention_mask_from_ids(token_ids, pad_token_id):
    mask = [1 if token_id != pad_token_id else 0 for token_id in token_ids]
    return torch.tensor(mask, dtype=torch.long).unsqueeze(0).unsqueeze(1).unsqueeze(2) # [batch_size, 1, 1, seq_len]

In [21]:
def prepare_transformer_inputs(input_text, vocab, max_seq_len):
    pad_token_id = vocab['<pad>']

    # Step 1: Add special tokens and assign initial segment IDs
    tokens, segment_ids = _add_special_tokens_and_segments(input_text, vocab)

    # Step 2: Convert tokens to IDs
    token_ids = _convert_to_ids(tokens, vocab)

    # Step 3: Pad token IDs and segment IDs
    padded_token_ids, padded_segment_ids = _pad_ids_and_segments(
        token_ids, segment_ids, max_seq_len, pad_token_id
    )

    # Convert to PyTorch tensors and add batch dimension
    input_ids = torch.tensor(padded_token_ids, dtype=torch.long).unsqueeze(0)
    token_type_ids = torch.tensor(padded_segment_ids, dtype=torch.long).unsqueeze(0)

    # Step 4: Create attention mask
    attention_mask = _create_attention_mask_from_ids(padded_token_ids, pad_token_id)

    return input_ids, attention_mask, token_type_ids

### Demonstration of the new tokenization pipeline

In [22]:


input_text = "anugrah is learning"
max_seq_len = 10 # maximum token need to be created

# Prepare inputs using the new modular function
input_ids, attention_mask_new, token_type_ids_new = prepare_transformer_inputs(
    input_text, new_vocab, max_seq_len
)

print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids}")
print(f"Attention Mask: {attention_mask_new}")
print(f"Token Type IDs: {token_type_ids_new}")

print(f"Input IDs shape: {input_ids.shape}")
print(f"Attention Mask shape: {attention_mask_new.shape}")
print(f"Token Type IDs shape: {token_type_ids_new.shape}")

Input Text: 'anugrah is learning'
Input IDs: tensor([[ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0]])
Attention Mask: tensor([[[[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]]]])
Token Type IDs: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Input IDs shape: torch.Size([1, 10])
Attention Mask shape: torch.Size([1, 1, 1, 10])
Token Type IDs shape: torch.Size([1, 10])


### Create and apply embedding

In [23]:
d_model = 8 #vector dimension for each word

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

embedded_input = embedding(input_ids)
print(f"Shape of embedded input: {embedded_input.shape}")

Shape of embedded input: torch.Size([1, 10, 8])


In [24]:
print(embedded_input)

tensor([[[-1.4619e+00, -1.1343e-01,  2.3175e+00,  9.1168e-01,  1.2581e+00,
          -2.8501e-01, -1.6501e+00,  1.9048e+00],
         [ 1.9438e-01, -5.0872e-01, -8.6753e-01,  1.0811e+00, -1.1352e+00,
           7.0629e-01, -1.0929e+00,  2.4185e-01],
         [ 9.0966e-01,  1.3300e+00,  8.4139e-03,  2.8730e-01,  2.2684e-01,
          -1.1336e+00,  5.9704e-01,  2.1985e-01],
         [ 5.0047e-01,  2.7760e-01, -4.7338e-01,  7.4079e-01, -1.6886e+00,
          -2.4873e+00,  1.1690e+00,  5.3599e-01],
         [-1.4619e+00, -1.1343e-01,  2.3175e+00,  9.1168e-01,  1.2581e+00,
          -2.8501e-01, -1.6501e+00,  1.9048e+00],
         [ 7.6889e-02,  1.2195e+00,  2.6104e+00,  7.0984e-01,  5.4182e-04,
          -2.5990e-02,  6.1414e-01, -1.8515e+00],
         [ 7.6889e-02,  1.2195e+00,  2.6104e+00,  7.0984e-01,  5.4182e-04,
          -2.5990e-02,  6.1414e-01, -1.8515e+00],
         [ 7.6889e-02,  1.2195e+00,  2.6104e+00,  7.0984e-01,  5.4182e-04,
          -2.5990e-02,  6.1414e-01, -1.8515e+00],


In [25]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nEncodings for each token in 'anugrah is learning':")
# Iterate through the input_ids and corresponding embedded vectors
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    embedding_vector = embedded_input.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Embedding: {embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Encodings for each token in 'anugrah is learning':
Token: '<unk>' (ID: 1) -> Embedding: [-1.4619050025939941, -0.11343101412057877, 2.317460298538208, 0.9116841554641724, 1.25811767578125, -0.28500720858573914, -1.6501322984695435, 1.904799461364746]
Token: 'anugrah' (ID: 38) -> Embedding: [0.19437795877456665, -0.5087205767631531, -0.8675291538238525, 1.0810924768447876, -1.1351792812347412, 0.7062867879867554, -1.0928802490234375, 0.2418464869260788]
Token: 'is' (ID: 5) -> Embedding: [0.9096562266349792, 1.3300310373306274, 0.00841390248388052, 0.28729724884033203, 0.22683921456336975, -1.133610486984253, 0.597036600112915, 0.2198544442653656]
Token: 'learning' (ID: 9) -> Embedding: [0.5004652142524719, 0.2775985598564148, -0.4733809530735016, 0.7407932877540588, -1.6886414289474487, -2.4872918128967285, 1.1689813137054443, 0.5359873175621033]
Token: '<unk>' (ID: 1) -> Embedding: [-1.461905

In [26]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super().__init__()

        self.dropout = nn.Dropout(0.1)

        # Create a positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term) if d_model % 2 == 0 else torch.cos(position * div_term[:-1])

        pe = pe.unsqueeze(0) # Add batch dimension
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Add positional encoding to the input embeddings
        # x has shape (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Instantiate Positional Encoding
pos_encoder = PositionalEncoding(d_model, max_seq_len)

# Apply positional encoding to the embedded input
embedded_with_pos = pos_encoder(embedded_input)

print(f"Shape of embedded input with positional encoding: {embedded_with_pos.shape}")

Shape of embedded input with positional encoding: torch.Size([1, 10, 8])


In [27]:
print(embedded_with_pos[0][1],embedded_with_pos.shape) #PE for anugrah (1,batch,10 total , 8 dimension)

tensor([ 1.1509,  0.0351, -0.8530,  2.3068, -1.2502,  1.8958, -1.2132,  1.3798],
       grad_fn=<SelectBackward0>) torch.Size([1, 10, 8])


In [28]:
print(f"Input Text: '{input_text}'")
print(f"Input IDs: {input_ids.squeeze(0)}")

print("\nCombined Embeddings (Word Embedding + Positional Encoding) for each token:")
for i, token_id in enumerate(input_ids.squeeze(0)):
    word = idx_to_word.get(token_id.item(), '<unknown>')
    combined_embedding_vector = embedded_with_pos.squeeze(0)[i]
    print(f"Token: '{word}' (ID: {token_id.item()}) -> Combined Embedding: {combined_embedding_vector.tolist()}")

Input Text: 'anugrah is learning'
Input IDs: tensor([ 1, 38,  5,  9,  1,  0,  0,  0,  0,  0])

Combined Embeddings (Word Embedding + Positional Encoding) for each token:
Token: '<unk>' (ID: 1) -> Combined Embedding: [-1.624338984489441, 0.9850766658782959, 2.574955940246582, 2.124093532562256, 1.3979085683822632, 0.7944364547729492, -1.8334804773330688, 3.2275550365448]
Token: 'anugrah' (ID: 38) -> Combined Embedding: [1.1509432792663574, 0.035090845078229904, -0.8529952764511108, 2.306774139404297, -1.2501994371414185, 1.895818829536438, -1.213200330734253, 1.379828929901123]
Token: 'is' (ID: 5) -> Combined Embedding: [2.02105975151062, 1.0154268741607666, 0.23009246587753296, 1.4081820249557495, 0.0, -0.14867830276489258, 0.6655962467193604, 0.0]
Token: 'learning' (ID: 9) -> Combined Embedding: [0.7128725051879883, -0.7915488481521606, -0.1976230889558792, 1.8845887184143066, -1.842940092086792, -1.6530463695526123, 1.302201509475708, 1.7066476345062256]
Token: '<unk>' (ID: 1) -> Com

In [29]:
token_type_embedding = nn.Embedding(2, d_model) # Assuming 2 types: 0 for sentence A, 1 for sentence B

'''
Segment Embedding

Tells the model which sentence a token belongs to.

Example:

Sentence A: I love AI
Sentence B: It is powerful

BERT input:

[CLS] I love AI [SEP] It is powerful [SEP]
'''

# Get segment embeddings for our token_type_ids
segment_embeddings = token_type_embedding(token_type_ids_new)

# Add the segment embeddings to the combined word and positional embeddings
final_transformer_input = embedded_with_pos + segment_embeddings

print(f"Shape of token_type_ids: {token_type_ids_new.shape}")
print(f"Shape of segment embeddings: {segment_embeddings.shape}")
print(f"Shape of final input to Transformer: {final_transformer_input.shape}")

print("\nSegment Embeddings:")
print(segment_embeddings)

Shape of token_type_ids: torch.Size([1, 10])
Shape of segment embeddings: torch.Size([1, 10, 8])
Shape of final input to Transformer: torch.Size([1, 10, 8])

Segment Embeddings:
tensor([[[ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
          -0.0886],
         [ 1.0443,  0.4038,  1.5990,  0.8096, -0.7387, -0.4374, -0.1904,
     

In [30]:
print('final transformer input')
print(final_transformer_input)

final transformer input
tensor([[[-0.5801,  1.3888,  4.1739,  2.9336,  0.6592,  0.3570, -2.0238,
           3.1390],
         [ 2.1952,  0.4388,  0.7460,  3.1163, -1.9889,  1.4584, -1.4036,
           1.2913],
         [ 3.0653,  1.4192,  1.8291,  2.2177, -0.7387, -0.5861,  0.4752,
          -0.0886],
         [ 1.7571, -0.3878,  1.4013,  2.6941, -2.5816, -2.0905,  1.1118,
           1.6181],
         [-1.4210, -0.4485,  4.6066,  2.8459, -0.7387,  0.3561, -2.0194,
           3.1390],
         [ 0.0642,  2.0739,  1.5990,  0.8096, -0.6826,  0.6434, -0.1904,
          -1.0346],
         [ 0.8192,  2.8256,  5.1267,  2.5153, -0.7387,  0.6428,  0.4987,
          -0.0886],
         [ 1.8597,  2.5964,  5.2152,  2.4481, -0.6604,  0.6421,  0.4998,
          -1.0347],
         [ 2.2290,  1.5971,  5.2964,  2.3724, -0.6493,  0.6412,  0.5009,
          -1.0347],
         [ 1.5876,  0.7464,  5.3697,  2.2889, -0.6382, -0.4374,  0.5020,
          -1.0347]]], grad_fn=<AddBackward0>)


In [31]:
import torch
import torch.nn as nn
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden_dim, dropout_rate=0.1):
        super().__init__()

        '''
          d_model = 512
          num_heads = 8

          head_dim = 64
          512 dimensions
          ├─ Head 1 : 64
          ├─ Head 2 : 64
          ├─ Head 3 : 64
          ├─ Head 4 : 64
          ├─ Head 5 : 64
          ├─ Head 6 : 64
          ├─ Head 7 : 64
          └─ Head 8 : 64
        '''

        assert d_model % num_heads == 0 #if it is divisible then return true and allow run

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Multi-Head Attention
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after MHA residual
        self.layer_norm1 = nn.LayerNorm(d_model)

        # Feed Forward Network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, d_model)
        )

        self.ffn_dropout = nn.Dropout(dropout_rate)

        # LayerNorm after FFN residual
        self.layer_norm2 = nn.LayerNorm(d_model)

    def _calculate_attention(self, Q, K, V, mask=None):

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        ) / math.sqrt(self.head_dim) # The division by sqrt(head_dim) is the scaling factor.

        if mask is not None:
            # Apply mask to the scores. Masked positions should be set to -inf.
            # This code is used to hide certain tokens from attention.1allowed 0 not allowed
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        output = torch.matmul(
            attention_weights,
            V
        )

        return output

    def forward(self, x, mask=None):
        # In your current setup (cell cbe1d5c1), `mask` is indeed passed as a tensor
        # (attention_mask_new), not None, when calling this layer.

        batch_size, seq_len, _ = x.shape

        # ----------------------------
        # Multi-Head Self Attention
        # ----------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        '''
        Full token embedding (8 dims)
                  │
                  ▼
            Split into 2 heads
                  │
          ┌──────┴──────┐
          ▼             ▼
        Head 1        Head 2
        (4 dims)      (4 dims)
          │             │
        Attention    Attention
        separately   separately
        '''

        '''
        # Split Q into multiple attention heads and rearrange dimensions
        # From: [batch_size, seq_len, d_model]
        # To:   [batch_size, num_heads, seq_len, head_dim]
        '''
        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_output = self._calculate_attention(
            Q,
            K,
            V,
            mask=mask
        )
        '''
        Before attention:

        Token
        [1,2,3,4,5,6,7,8]

        ↓ split

        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        After attention:
        Head1 [1,2,3,4]
        Head2 [5,6,7,8]

        ↓ concatenate
        combined attention heads : [1,2,3,4,5,6,7,8]
        '''
        attention_output = (
            attention_output
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model
            )
        )
        '''
        Multi-head outputs
              ↓
        Concatenate
              ↓
        Wo (Linear Layer)
              ↓
        Final attention representation

        [1,2,3,4,5,6,7,8]
            ↓
        Linear Wo
            ↓
       [5,-2,8,1,4,9,3,7]
        '''

        mha_output = self.Wo(
            attention_output
        )
        '''
        Some output values are randomly set to 0.Prevent Overfitting

        Before:
        [5,-2,8,1,4,9,3,7]

        After dropout:
        [5,0,8,1,0,9,3,7]
        '''
        final_mha_output = self.attn_dropout(
            mha_output
        )

        '''
          Input
            ↓
          Wq, Wk, Wv
            ↓
          Split into heads
            ↓
          Attention
            ↓
          Merge heads
            ↓
          Wo
            ↓
          Dropout
            ↓
          Final MHA output
        '''

        # Residual Connection + LayerNorm
        x = self.layer_norm1(
            x + final_mha_output
        )

        # ----------------------------
        # Feed Forward Network
        # ----------------------------

        ffn_output = self.ffn(x)

        ffn_output = self.ffn_dropout(
            ffn_output
        )

        # Residual Connection + LayerNorm
        output = self.layer_norm2(
            x + ffn_output
        )

        return output

# --- Execution Code ---
encoder_layer = TransformerEncoderLayer(d_model=8, num_heads=2, ffn_hidden_dim=32)
encoder_output = encoder_layer(final_transformer_input, mask=attention_mask_new)

print(f"Encoder Layer Input Shape: {final_transformer_input.shape}")
print(f"Encoder Layer Output Shape: {encoder_output.shape}")
print("\nFirst few values of the output:")
print(encoder_output[0, 0, :])

Encoder Layer Input Shape: torch.Size([1, 10, 8])
Encoder Layer Output Shape: torch.Size([1, 10, 8])

First few values of the output:
tensor([-0.6930,  0.0591,  1.1295,  0.8939, -0.6892,  0.1002, -1.9228,  1.1222],
       grad_fn=<SelectBackward0>)


In [32]:
# --- Transformer Classifier Model Definition (re-defined as it was deleted) ---
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, num_classes, dropout_rate=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len)
        self.token_type_embedding = nn.Embedding(2, d_model) # Assuming 2 types for segment embedding

        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, ffn_hidden_dim, dropout_rate)
            for _ in range(num_layers)
        ])

        # Classification head: takes the CLS token output and classifies
        self.classification_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, num_classes) # Output `num_classes` logits
        )

    def forward(self, input_ids, token_type_ids, attention_mask):
        # Embeddings
        x = self.embedding(input_ids)
        x = self.positional_encoding(x)
        segment_embeddings = self.token_type_embedding(token_type_ids)
        x = x + segment_embeddings

        # Pass through encoder layers
        for layer in self.encoder_layers:
            # attention_mask here should be [batch_size, 1, 1, seq_len]
            # It will broadcast to [batch_size, num_heads, seq_len, seq_len] within the layer.
            x = layer(x, mask=attention_mask)

        # For classification, we typically take the output corresponding to the [CLS] token
        # The [CLS] token is the first token in the sequence (index 0)
        cls_token_output = x[:, 0, :]

        # Pass through classification head
        logits = self.classification_head(cls_token_output)
        return logits.squeeze(1) if num_classes == 1 else logits # Squeeze for binary classification

In [33]:
from torch.utils.data import Dataset

# --- TextDataset for Training Demonstration (re-defined as it was deleted) ---
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_seq_len):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        input_ids_batch_dim, attention_mask_batch_dim, token_type_ids_batch_dim = prepare_transformer_inputs(text, self.vocab, self.max_seq_len)

        input_ids = input_ids_batch_dim.squeeze(0)
        token_type_ids = token_type_ids_batch_dim.squeeze(0)
        attention_mask_for_encoder = attention_mask_batch_dim.squeeze(1) # Shape: [1, 1, seq_len]

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask_for_encoder,
            'token_type_ids': token_type_ids,
            'labels': torch.tensor(label, dtype=torch.float)
        }

In [37]:
from torch.utils.data import DataLoader
import torch.optim as optim

# --- Data Preparation for Training ---
dummy_texts = [
    "this is a positive sentence",
    "i hate this movie",
    "learning transformers is fun",
    "what a terrible experience",
    "feeling very happy today",
    "the product is excellent!",
    "absolutely dreadful service",
    "i really enjoyed the movie",
    "a wonderful day for a walk",
    "i am so disappointed",
    "the weather is beautiful today",
    "this restaurant has amazing food and great service",
    "i had a terrible day at work",
    "such a joyful and uplifting experience",
    "the book was incredibly boring and slow",
    "cannot recommend this enough, truly fantastic",
    "utterly disgusted by the lack of professionalism",
    "a truly inspiring performance by the actor",
    "this journey was frustrating and long",
    "my expectations were completely exceeded",
    "the worst customer support I have ever encountered",
    "feeling optimistic about the future",
    "a complete waste of time and money",
    "this made me smile, so heartwarming",
    "the situation is absolutely hopeless",
    "This movie was a masterpiece, I highly recommend it.",
    "The customer service was outstanding, very helpful and friendly.",
    "A truly delightful experience from start to finish.",
    "I'm very impressed with the results, well done!",
    "Such a beautiful day, full of joy.",
    "Completely ruined my evening, a disaster.",
    "The product broke after one use, very disappointed.",
    "I would not recommend this to anyone, save your money.",
    "This situation is incredibly frustrating and unacceptable.",
    "My expectations were not met at all, quite poor."
]

dummy_labels = [
    1, 0, 1, 0, 1, 1, 0, 1, 1, 0,
    1, 1, 0, 1, 0, 1, 0, 1, 0, 1,
    0, 1, 0, 1, 0,
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0
] # 1 for positive/neutral, 0 for negative

# Re-using existing `new_vocab`, `vocab_size` from kernel state

# --- Hyperparameters ---
d_model = 8
num_heads = 2
ffn_hidden_dim = 32
num_layers = 2
max_seq_len = 12 # Adjusted to accommodate longer sentences
num_classes = 1 # For binary classification (sentiment: positive/negative)
epochs = 5
batch_size = 2
learning_rate = 0.001

# Instantiate the dataset and DataLoader
dataset = TextDataset(dummy_texts, dummy_labels, new_vocab, max_seq_len)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Instantiate the Transformer Classifier model
model = TransformerClassifier(vocab_size, d_model, num_heads, ffn_hidden_dim, num_layers, max_seq_len, num_classes)

# Loss and Optimizer
criterion = nn.BCEWithLogitsLoss() # For binary classification with logits
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# --- Training Loop ---
print(f"\nStarting training for {epochs} epochs...")
model.train()
for epoch in range(epochs):
    total_loss = 0
    total_correct = 0
    total_samples = 0

    for i, batch in enumerate(data_loader):
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        token_type_ids = batch['token_type_ids']
        labels = batch['labels']

        optimizer.zero_grad()

        # Forward pass
        logits = model(input_ids, token_type_ids, attention_mask) # logits are raw scores
        loss = criterion(logits, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calculate accuracy
        predictions = (torch.sigmoid(logits) > 0.5).long() # Convert logits to binary predictions
        total_correct += (predictions == labels.long()).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = (total_correct / total_samples) * 100
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

print("Training complete!")

# --- Demonstration of the trained model (Inference) ---
model.eval()
print("\n--- Inference Examples ---")
def predict_sentiment(model, text, vocab, max_seq_len):
    input_ids, attention_mask_prep, token_type_ids = prepare_transformer_inputs(text, vocab, max_seq_len)
    attention_mask_for_model = attention_mask_prep.squeeze(1)

    with torch.no_grad():
        logits = model(input_ids, token_type_ids, attention_mask_for_model)
        prediction = torch.sigmoid(logits).item()
    return "Positive/Neutral" if prediction > 0.5 else "Negative", prediction

# Test new sentences
new_sentence_1 = "this is a really great day"
pred_label_1, pred_score_1 = predict_sentiment(model, new_sentence_1, new_vocab, max_seq_len)
print(f"Sentence: '{new_sentence_1}'\nPrediction: {pred_label_1} (Score: {pred_score_1:.4f})")

new_sentence_2 = "i am very disappointed with the quality"
pred_label_2, pred_score_2 = predict_sentiment(model, new_sentence_2, new_vocab, max_seq_len)
print(f"Sentence: '{new_sentence_2}'\nPrediction: {pred_label_2} (Score: {pred_score_2:.4f})")

new_sentence_3 = "learning about transformers is so cool"
pred_label_3, pred_score_3 = predict_sentiment(model, new_sentence_3, new_vocab, max_seq_len)
print(f"Sentence: '{new_sentence_3}'\nPrediction: {pred_label_3} (Score: {pred_score_3:.4f})")


Starting training for 5 epochs...
Epoch 1/5, Loss: 0.7239, Accuracy: 54.29%
Epoch 2/5, Loss: 0.6932, Accuracy: 54.29%
Epoch 3/5, Loss: 0.6920, Accuracy: 54.29%
Epoch 4/5, Loss: 0.6969, Accuracy: 54.29%
Epoch 5/5, Loss: 0.6837, Accuracy: 54.29%
Training complete!

--- Inference Examples ---
Sentence: 'this is a really great day'
Prediction: Positive/Neutral (Score: 0.5773)
Sentence: 'i am very disappointed with the quality'
Prediction: Positive/Neutral (Score: 0.5715)
Sentence: 'learning about transformers is so cool'
Prediction: Positive/Neutral (Score: 0.5699)
